In [47]:
pip install ollama pydantic geojson chromadb


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /opt/jupyter/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import requests
import chromadb
from typing import Optional, Dict, Any, List
from pydantic import BaseModel
from ollama import Client

In [49]:
OLLAMA_HOST = "http://10.10.10.100:11434"

EMBED_MODEL = "nomic-embed-text-v2-moe:latest"

LLM_MODEL = (
"gemma4:latest"
)
client = Client(
    host=OLLAMA_HOST
)

print("Configuration Loaded")

Configuration Loaded


In [50]:
class HighwayFeature(BaseModel):

    fid: Optional[int]

    objectid: Optional[int]

    alignment: Optional[str]

    year_rec: Optional[int]

    state_code: Optional[int]

    route_id: Optional[str]

    route: Optional[str]

    comments: Optional[str]

    rte_geomid: Optional[str]

    rte_ver_id: Optional[str]

    direction: Optional[str]

    shape_leng: Optional[float]

    shape_le_1: Optional[float]

    shape_length: Optional[float]

    geometry: Optional[Dict[str, Any]]

In [51]:
URL = (
    "https://services.arcgis.com/"
    "V6ZHFr6zdgNZuVG0/arcgis/rest/services/"
    "California_Highways/FeatureServer/0/query"
)

features = []

offset = 0

batch_size = 2000

while True:

    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": True,
        "f": "json",
        "resultOffset": offset,
        "resultRecordCount": batch_size
    }

    response = requests.get(
        URL,
        params=params,
        timeout=120
    )

    data = response.json()

    batch = data.get(
        "features",
        []
    )

    if not batch:
        break

    features.extend(batch)

    offset += batch_size

    print(
        f"Downloaded {len(features)}"
    )

print(
    f"\nTotal Features: {len(features)}"
)

Downloaded 508

Total Features: 508


In [52]:


def feature_to_model(
    feature
) -> HighwayFeature:

    attrs = feature["attributes"]

    return HighwayFeature(
        fid=attrs.get("FID"),
        objectid=attrs.get("OBJECTID"),
        alignment=attrs.get("ALIGNMENT"),
        year_rec=attrs.get("YEAR_REC"),
        state_code=attrs.get("STATE_CODE"),
        route_id=attrs.get("ROUTE_ID"),
        route=attrs.get("ROUTE"),
        comments=attrs.get("COMMENTS"),
        rte_geomid=attrs.get("RTE_GEOMID"),
        rte_ver_id=attrs.get("RTE_VER_ID"),
        direction=attrs.get("DIR"),
        shape_leng=attrs.get("Shape_Leng"),
        shape_le_1=attrs.get("SHAPE_Le_1"),
        shape_length=attrs.get("Shape__Length"),
        geometry=feature.get("geometry")
    )

In [53]:
highways = [
    feature_to_model(f)
    for f in features
]

print(
    highways[0]
)

fid=1 objectid=1 alignment='L' year_rec=2015 state_code=6 route_id='101L' route='101' comments=' ' rte_geomid='101_20180226_L' rte_ver_id='101_2016_12' direction='S' shape_leng=0.0 shape_le_1=12.9911025579 shape_length=12.991102555174564 geometry={'paths': [[[-118.221264725387, 34.0340850641744], [-118.221365697669, 34.0342486958207], [-118.221469812182, 34.0344222973512], [-118.221543804803, 34.0346571795852], [-118.22159380531, 34.0348091811988], [-118.221664804987, 34.0350641803677], [-118.22170280494, 34.0352351810588], [-118.221734804617, 34.0354071808968], [-118.221758804825, 34.0355811808269], [-118.221779804894, 34.035824181241], [-118.221786806116, 34.0361031806173], [-118.221781804986, 34.0362771814468], [-118.221759804871, 34.0365381808924], [-118.221692805379, 34.0369141811446], [-118.221621804802, 34.0373511806126], [-118.221596805448, 34.0375401812344], [-118.221574805333, 34.0378251808875], [-118.221570806048, 34.0380151806561], [-118.221573805287, 34.0382051804248], [-1

In [54]:
def model_to_text(
    h: HighwayFeature
):

    return json.dumps(
        {
            "layer":
                "CA_Highways",

            "route":
                h.route,

            "route_id":
                h.route_id,

            "direction":
                h.direction,

            "year":
                h.year_rec,

            "state_code":
                h.state_code,

            "comments":
                h.comments,

            "geometry":
                h.geometry,

            "description":
                (
                    f"California State "
                    f"Route {h.route} "
                    f"travelling "
                    f"{h.direction}bound"
                )
        },
        default=str
    )

In [55]:
chroma_client = (
    chromadb.PersistentClient(
        path="./chroma"
    )
)

collection = (
    chroma_client
    .get_or_create_collection(
        "ca_highways"
    )
)

print(
    "Collection Ready"
)

Collection Ready


In [56]:
for h in highways:

    text = model_to_text(h)

    embedding = client.embed(
        model=EMBED_MODEL,
        input=text
    )["embeddings"][0]

    collection.add(
        ids=[
            str(h.objectid)
        ],

        embeddings=[
            embedding
        ],

        documents=[
            text
        ],

        metadatas=[
            {
                "route":
                    str(h.route),

                "direction":
                    str(h.direction)
            }
        ]
    )

print(
    f"Inserted {len(highways)} records"
)

Inserted 508 records


In [57]:
routes = sorted(
    {
        h.route
        for h in highways
    }
)

print(
    "5" in routes
)

print(
    routes[:50]
)

True
['1', '10', '101', '101U', '103', '104', '105', '107', '108', '109', '10S', '11', '110', '111', '112', '113', '114', '115', '116', '118', '119', '12', '120', '121', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '14', '140', '142', '144', '145', '146', '147', '149']


In [58]:
question = (
    "Tell me about Route 5"
)

query_embedding = client.embed(
    model=EMBED_MODEL,
    input=question
)["embeddings"][0]

results = collection.query(
    query_embeddings=[
        query_embedding
    ],
    n_results=5
)

print(
    results["documents"][0]
)

['{"layer": "CA_Highways", "route": "5", "route_id": "5R", "direction": "N", "year": 2015, "state_code": 6, "comments": " ", "geometry": {"paths": [[[-117.03243597584, 32.5444843461757], [-117.033259218838, 32.5450019662678], [-117.03451795404, 32.5458205004158], [-117.035567294695, 32.5464915070775], [-117.036892656171, 32.5473552510441], [-117.038686768579, 32.5484926344276], [-117.039724214801, 32.549182677039], [-117.040582599707, 32.5496831767373], [-117.041127926113, 32.5499319274166], [-117.042228823102, 32.5503287901416], [-117.04527000021, 32.5514383925678], [-117.04573882129, 32.5516296423941], [-117.048274872587, 32.5527124117485], [-117.059019872569, 32.5572514124228], [-117.062344122983, 32.5587048904184], [-117.062637873039, 32.5588284123015], [-117.063753873243, 32.5593254118384], [-117.064428026632, 32.5596804183167], [-117.065584873143, 32.5603804146245], [-117.067505978818, 32.561589484767], [-117.069286872991, 32.5627514151457], [-117.072433079223, 32.5647084082665],

In [59]:
context = "\n\n".join(
    results["documents"][0]
)

prompt = f"""
You are a California GIS analyst.

Answer ONLY from the supplied context.

If information is missing,
say so.

Context:

{context}

Question:

{question}

Answer:
"""

In [60]:
response = client.generate(
    model=LLM_MODEL,
    prompt=prompt
)

print(
    response["response"]
)

You provided data for two distinct routes: **Route 1** (implied by the first set of coordinates) and **Route 2** (labeled as a segment of State Route 2).

Since you asked for information about "Route 1" and "Route 2" generally, I will summarize the information available for both segments.

***

### 🗺️ Route 1 (Implied Route)

*   **General Observation:** This route segment appears to traverse a relatively straight path through a mix of developed and semi-developed areas, suggesting a primary connecting road.
*   **Start Point:** (Approximate) Near the beginning of the provided coordinates.
*   **End Point:** (Approximate) At the final coordinates provided.

### 🛣️ Route 2 (State Route 2 Segment)

*   **Route Designation:** This segment is part of **State Route 2**.
*   **General Observation:** This portion of the route is characterized by a sustained path through what appears to be rolling or semi-rural terrain, with steady progression over a significant distance.
*   **Geographical Fo

In [61]:
def ask(question):

    query_embedding = (
        client.embed(
            model=EMBED_MODEL,
            input=question
        )["embeddings"][0]
    )

    results = collection.query(
        query_embeddings=[
            query_embedding
        ],
        n_results=5
    )

    context = "\n\n".join(
        results["documents"][0]
    )

    prompt = f"""
You are a California GIS analyst.

Use ONLY the supplied context.

Context:

{context}

Question:

{question}

Answer:
"""

    response = client.generate(
        model=LLM_MODEL,
        prompt=prompt
    )

    return response["response"]

In [62]:
print(
    ask(
        "Tell me about Route 101"
    )
)

I do not have specific information about Route 10 within the context of the provided geographical data, which details a specific route (likely a segment of US 101 based on the provided coordinates/context) and its associated geography.

If you can provide the specific location or context you are interested in for Route 10, I would be happy to try and help you!
